# PDF 문서 로더

PDF(Portable Document Format)는 가장 널리 사용되는 문서 형식 중 하나입니다.

LangChain은 다양한 PDF 파서를 제공하며, 각각의 특징과 장단점이 있습니다. 이 노트북에서는 주요 PDF 로더들을 비교하고 상황에 맞는 선택 방법을 학습합니다.


## 학습 목표

- 주요 PDF 로더 (PyPDF, PyMuPDF, PDFPlumber) 비교
- 각 로더의 특징과 적합한 사용 사례 이해
- OCR 기능이 있는 PDF 로더 활용법 학습


## PDF 로더 선택 가이드

### 상황별 추천

| 상황 | 추천 로더 | 이유 |
|------|----------|------|
| **일반적인 PDF** | PyPDF | 가볍고 빠르며, 대부분의 경우에 충분 |
| **복잡한 레이아웃** | PDFPlumber | 테이블, 레이아웃 정보 보존 |
| **속도가 중요한 경우** | PyMuPDF | 가장 빠른 처리 속도 |
| **이미지 속 텍스트** | PyPDF + OCR | 스캔 문서나 이미지 텍스트 추출 |
| **한글 문서** | PDFMiner, PDFPlumber | 한글 인식률이 상대적으로 우수 |

### 성능 비교 요약

- **속도**: PyMuPDF > PyPDF > PDFPlumber
- **정확도**: PDFPlumber ≥ PDFMiner > PyPDF
- **기능**: PDFPlumber(테이블) > PyMuPDF(메타데이터) > PyPDF(기본)
- **사용 편의성**: PyPDF > PyMuPDF > PDFPlumber


> 🎯 **강의 포인트**: PDF 로더 선택은 "속도 vs 정확도" 트레이드오프입니다. PyMuPDF(빠름) → PDFPlumber(테이블 정확도) → PyPDF+OCR(이미지 텍스트) 순으로 상황에 맞게 선택하세요.

In [1]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()

# 예제 파일 경로
FILE_PATH = "./data/2026_gov.pdf"

# 데이터 파일 경로 확인
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {FILE_PATH}\n현재 디렉토리: {os.getcwd()}")

In [2]:
# 메타데이터 확인 헬퍼 함수
def show_metadata(docs):
    """Document의 메타데이터를 보기 좋게 출력"""
    if docs:
        print("=" * 60)
        print("📋 메타데이터")
        print("=" * 60)
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            # 값이 너무 길면 잘라서 표시
            v_str = str(v)
            if len(v_str) > 100:
                v_str = v_str[:100] + "..."
            print(f"{k:<{max_key_length}} : {v_str}")


## 1. PyPDF - 기본 PDF 로더

`PyPDF`는 LangChain에서 가장 기본적으로 사용되는 PDF 로더입니다.

### 주요 특징

- ✅ **가볍고 빠름**: 외부 의존성이 적어 설치와 사용이 간단
- ✅ **안정적**: 대부분의 표준 PDF에서 잘 작동
- ✅ **페이지 단위 처리**: 각 페이지가 별도의 Document 객체로 생성
- ❌ **레이아웃 보존 약함**: 복잡한 레이아웃이나 테이블 처리에는 부적합


In [3]:
from langchain_community.document_loaders import PyPDFLoader
import time

# ============================================================
# TODO: PyPDFLoader로 PDF를 로딩하고 시간을 측정해보세요
# 힌트: time.time()으로 시작/종료 시간을 기록합니다
# 예상 결과: 로딩 시간, 총 페이지 수, 내용 미리보기가 출력됩니다
# ============================================================

# 1단계: PyPDFLoader 초기화
# - PyPDFLoader: 가볍고 빠른 기본 PDF 파서, 페이지별 Document 생성
loader = PyPDFLoader(FILE_PATH)

# 2단계: 성능 측정을 포함한 로딩
start_time = time.time()
pypdf_docs = loader.load()
pypdf_time = time.time() - start_time

print("=" * 60)
print("📄 PyPDF 로딩 결과")
print("=" * 60)
print(f"로딩 시간: {pypdf_time:.3f}초")
print(f"총 페이지 수: {len(pypdf_docs)}")
print(f"\n첫 번째 페이지 내용 미리보기:")
print(pypdf_docs[0].page_content[:200])
print("...")

📄 PyPDF 로딩 결과
로딩 시간: 4.447초
총 페이지 수: 501

첫 번째 페이지 내용 미리보기:
발간등록번호
11-1053000-100001-09
이렇게 
달라집니다년부터2026
...


In [4]:
# PyPDF 메타데이터 확인
show_metadata(pypdf_docs)


📋 메타데이터
producer     : Adobe PDF Library 16.0.7
creator      : Adobe InDesign 17.3 (Macintosh)
creationdate : 2026-01-05T22:01:47+09:00
moddate      : 2026-01-05T22:01:47+09:00
title        : 
trapped      : /False
source       : ./data/2026_gov.pdf
total_pages  : 501
page         : 0
page_label   : 1


> 🔑 **핵심 개념**: PyMuPDF 메타데이터에는 `format`, `author`, `moddate` 등 PyPDF보다 훨씬 풍부한 정보가 포함됩니다. 문서 추적이 중요한 시스템에서는 이 메타데이터를 적극 활용하세요.

> ⚠️ **자주 하는 실수**: OCR(`extract_images=True`)은 처리 속도가 일반 모드보다 10배 이상 느릴 수 있습니다. 스캔 문서 여부를 먼저 확인하고 필요한 경우에만 활성화하세요.

### PyPDF + OCR (이미지 텍스트 추출)

스캔된 PDF나 이미지 속 텍스트를 추출해야 할 때는 OCR(Optical Character Recognition) 기능을 활성화할 수 있습니다.

**필요한 패키지**: `rapidocr-onnxruntime`

```bash
pip install rapidocr-onnxruntime
```


In [14]:
# OCR 기능 활성화 (이미지에서 텍스트 추출)
loader_with_ocr = PyPDFLoader(FILE_PATH, extract_images=True)
docs_with_ocr = loader_with_ocr.load()

print("💡 OCR 기능 사용법:")
print("PyPDFLoader(file_path, extract_images=True)")
print("\n✅ 언제 사용?")
print("  - 스캔된 PDF 문서")
print("  - 이미지로 된 텍스트가 포함된 PDF")
print("  - 표나 그래프 내의 텍스트 추출")
print("\n⚠️ 주의사항:")
print("  - OCR 처리는 시간이 오래 걸림")
print("  - rapidocr-onnxruntime 패키지 필요")


💡 OCR 기능 사용법:
PyPDFLoader(file_path, extract_images=True)

✅ 언제 사용?
  - 스캔된 PDF 문서
  - 이미지로 된 텍스트가 포함된 PDF
  - 표나 그래프 내의 텍스트 추출

⚠️ 주의사항:
  - OCR 처리는 시간이 오래 걸림
  - rapidocr-onnxruntime 패키지 필요


## 2. PyMuPDF - 고속 PDF 로더

`PyMuPDF`는 속도가 매우 빠르고 상세한 메타데이터를 제공하는 PDF 로더입니다.

### 주요 특징

- ⚡ **매우 빠른 속도**: C 라이브러리 기반으로 처리 속도가 빠름
- 📊 **풍부한 메타데이터**: 페이지 크기, 작성자, 생성일 등 상세 정보 제공
- ✅ **안정적인 한글 처리**: 한글 PDF도 잘 처리
- ❌ **큰 패키지 크기**: 설치 파일이 상대적으로 큼

**필요한 패키지**: `pymupdf`

```bash
pip install pymupdf
```


In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

# ============================================================
# TODO: PyMuPDFLoader로 같은 PDF를 로딩하고 PyPDF와 시간을 비교해보세요
# 힌트: PyMuPDFLoader(FILE_PATH)로 초기화합니다
# 예상 결과: PyPDF보다 빠른 속도와 더 많은 메타데이터 필드가 출력됩니다
# ============================================================

# 1단계: PyMuPDFLoader 초기화
# - PyMuPDFLoader: C 기반 라이브러리, 가장 빠른 속도, 풍부한 메타데이터
loader = PyMuPDFLoader(FILE_PATH)

# 2단계: 성능 측정
start_time = time.time()
pymupdf_docs = loader.load()
pymupdf_time = time.time() - start_time

print("=" * 60)
print("⚡ PyMuPDF 로딩 결과")
print("=" * 60)
print(f"로딩 시간: {pymupdf_time:.3f}초")
print(f"총 페이지 수: {len(pymupdf_docs)}")
print(f"PyPDF 대비 속도: {pypdf_time / pymupdf_time:.1f}배")
print(f"\n첫 번째 페이지 내용 미리보기:")
print(pymupdf_docs[0].page_content[:200])
print("...")

⚡ PyMuPDF 로딩 결과
로딩 시간: 0.921초
총 페이지 수: 501
PyPDF 대비 속도: 4.8배

첫 번째 페이지 내용 미리보기:
발
간
등
록
번
호
11-1053000-100001-09
이렇게 
달라집니다
년부터
20
26
...


In [7]:
# PyMuPDF 메타데이터 확인 (더 상세한 정보 제공)
show_metadata(pymupdf_docs)


📋 메타데이터
producer     : Adobe PDF Library 16.0.7
creator      : Adobe InDesign 17.3 (Macintosh)
creationdate : 2026-01-05T22:01:47+09:00
source       : ./data/2026_gov.pdf
file_path    : ./data/2026_gov.pdf
total_pages  : 501
format       : PDF 1.6
title        : 
author       : 
subject      : 
keywords     : 
moddate      : 2026-01-05T22:01:47+09:00
trapped      : 
modDate      : D:20260105220147+09'00'
creationDate : D:20260105220147+09'00'
page         : 0


## 3. PDFPlumber - 레이아웃 보존 로더

`PDFPlumber`는 복잡한 레이아웃과 테이블 정보를 보존하는 데 특화된 PDF 로더입니다.

### 주요 특징

- 📊 **테이블 추출 우수**: 표 구조를 잘 인식하고 보존
- 🎯 **정확한 텍스트 위치**: 텍스트의 정확한 위치 정보 제공
- ✅ **복잡한 레이아웃**: 다단 구성이나 복잡한 레이아웃도 처리
- ❌ **느린 속도**: 정확도를 위해 처리 시간이 더 소요됨

**필요한 패키지**: `pdfplumber`

```bash
pip install pdfplumber
```


In [8]:
from langchain_community.document_loaders import PDFPlumberLoader

# ============================================================
# TODO: PDFPlumberLoader로 PDF를 로딩하고 앞선 로더들과 비교해보세요
# 힌트: PDFPlumberLoader(FILE_PATH)로 초기화합니다
# 예상 결과: 로딩 시간과 페이지 수, 내용 미리보기가 출력됩니다
# ============================================================

# 1단계: PDFPlumberLoader 초기화
# - PDFPlumberLoader: 테이블/레이아웃 보존 특화, 정확도가 높지만 느림
loader = PDFPlumberLoader(FILE_PATH)

# 2단계: 성능 측정
start_time = time.time()
pdfplumber_docs = loader.load()
pdfplumber_time = time.time() - start_time

print("=" * 60)
print("📊 PDFPlumber 로딩 결과")
print("=" * 60)
print(f"로딩 시간: {pdfplumber_time:.3f}초")
print(f"총 페이지 수: {len(pdfplumber_docs)}")
print(f"\n첫 번째 페이지 내용 미리보기:")
print(pdfplumber_docs[0].page_content[:200])
print("...")

📊 PDFPlumber 로딩 결과
로딩 시간: 7.932초
총 페이지 수: 501

첫 번째 페이지 내용 미리보기:
발 간 등 록 번 호
11-1053000-100001-09
20
26
년부터
이렇게
달라집니다

...


In [9]:
# PDFPlumber 메타데이터 확인
show_metadata(pdfplumber_docs)


📋 메타데이터
source       : ./data/2026_gov.pdf
file_path    : ./data/2026_gov.pdf
page         : 0
total_pages  : 501
CreationDate : D:20260105220147+09'00'
Creator      : Adobe InDesign 17.3 (Macintosh)
ModDate      : D:20260105220147+09'00'
Producer     : Adobe PDF Library 16.0.7
Title        : 
Trapped      : False


## 4. 로더 비교 및 벤치마크

세 가지 PDF 로더를 실제로 비교해보겠습니다.


> 💡 **실무 팁**: 실제 프로젝트에서는 소량의 샘플 PDF로 세 로더를 먼저 테스트해보고 결과 텍스트 품질을 눈으로 비교하는 것이 가장 확실합니다. 같은 PDF도 로더마다 추출 결과가 눈에 띄게 다를 수 있습니다.

In [10]:
# ---------------------------------------------------
# 세 가지 PDF 로더 비교 결과 출력
# ---------------------------------------------------

# 1단계: 속도·페이지 수·글자 수 비교표 출력
print("=" * 60)
print("📊 PDF 로더 비교 결과")
print("=" * 60)
print(f"\n{'로더':<15} {'시간(초)':<12} {'페이지 수':<10} {'첫 페이지 글자 수'}")
print("-" * 60)
print(f"{'PyPDF':<15} {pypdf_time:<12.3f} {len(pypdf_docs):<10} {len(pypdf_docs[0].page_content)}")
print(f"{'PyMuPDF':<15} {pymupdf_time:<12.3f} {len(pymupdf_docs):<10} {len(pymupdf_docs[0].page_content)}")
print(f"{'PDFPlumber':<15} {pdfplumber_time:<12.3f} {len(pdfplumber_docs):<10} {len(pdfplumber_docs[0].page_content)}")

# 2단계: 속도 순위 정렬
print("\n⚡ 속도 순위:")
times = [
    ("PyPDF", pypdf_time),
    ("PyMuPDF", pymupdf_time),
    ("PDFPlumber", pdfplumber_time)
]
sorted_times = sorted(times, key=lambda x: x[1])
for i, (name, t) in enumerate(sorted_times, 1):
    print(f"  {i}. {name}: {t:.3f}초")

📊 PDF 로더 비교 결과

로더              시간(초)        페이지 수      첫 페이지 글자 수
------------------------------------------------------------
PyPDF           4.447        501        45
PyMuPDF         0.921        501        53
PDFPlumber      7.932        501        53

⚡ 속도 순위:
  1. PyMuPDF: 0.921초
  2. PyPDF: 4.447초
  3. PDFPlumber: 7.932초


In [11]:
# 텍스트 내용 비교 (첫 300자)
print("=" * 60)
print("📝 텍스트 추출 결과 비교 (첫 300자)")
print("=" * 60)

print("\n[PyPDF]")
print(pypdf_docs[0].page_content[:300])
print("\n" + "-" * 60)

print("\n[PyMuPDF]")
print(pymupdf_docs[0].page_content[:300])
print("\n" + "-" * 60)

print("\n[PDFPlumber]")
print(pdfplumber_docs[0].page_content[:300])


📝 텍스트 추출 결과 비교 (첫 300자)

[PyPDF]
발간등록번호
11-1053000-100001-09
이렇게 
달라집니다년부터2026

------------------------------------------------------------

[PyMuPDF]
발
간
등
록
번
호
11-1053000-100001-09
이렇게 
달라집니다
년부터
20
26

------------------------------------------------------------

[PDFPlumber]
발 간 등 록 번 호
11-1053000-100001-09
20
26
년부터
이렇게
달라집니다



## 5. 실전 활용 가이드

### 시나리오별 최적 로더 선택

#### 시나리오 1: 대량의 표준 PDF 문서 처리
```python
# PyMuPDF 사용 - 속도가 가장 빠름
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("large_document.pdf")
docs = loader.load()
```

**이유**: 수백~수천 개의 PDF를 처리해야 할 때는 속도가 가장 중요합니다.

#### 시나리오 2: 복잡한 테이블이 포함된 보고서
```python
# PDFPlumber 사용 - 테이블 구조 보존
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("financial_report.pdf")
docs = loader.load()
```

**이유**: 재무제표, 통계 보고서 등 테이블 정보가 중요한 경우 정확도가 높습니다.

#### 시나리오 3: 스캔된 문서 (이미지 PDF)
```python
# PyPDF + OCR 사용
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("scanned_document.pdf", extract_images=True)
docs = loader.load()
```

**이유**: 텍스트 레이어가 없는 스캔 문서는 OCR이 필수입니다.

#### 시나리오 4: 일반적인 사용 (기본 선택)
```python
# PyPDF 사용 - 가볍고 안정적
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("standard_document.pdf")
docs = loader.load()
```

**이유**: 특별한 요구사항이 없다면 PyPDF가 가장 무난한 선택입니다.


## 6. 고급 기능

### 페이지 범위 지정

특정 페이지만 로드하고 싶을 때는 `lazy_load()`를 사용하여 필터링할 수 있습니다.


In [12]:
# 특정 페이지만 로드 (3~5페이지)
loader = PyPDFLoader(FILE_PATH)

# lazy_load()로 제너레이터 방식으로 순회하며 필터링
selected_pages = []
for i, doc in enumerate(loader.lazy_load()):
    if 2 <= i <= 4:  # 0-based index: 3~5페이지
        selected_pages.append(doc)
    if i > 4:
        break

print("=" * 60)
print("📄 선택된 페이지")
print("=" * 60)
print(f"로드된 페이지: {len(selected_pages)}개")
for i, doc in enumerate(selected_pages, start=3):
    print(f"\n페이지 {i}:")
    print(f"  메타데이터: {doc.metadata}")
    print(f"  내용 길이: {len(doc.page_content)} 글자")


📄 선택된 페이지
로드된 페이지: 3개

페이지 3:
  메타데이터: {'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 2, 'page_label': '3'}
  내용 길이: 109 글자

페이지 4:
  메타데이터: {'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 3, 'page_label': '4'}
  내용 길이: 72 글자

페이지 5:
  메타데이터: {'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Macintosh)', 'creationdate': '2026-01-05T22:01:47+09:00', 'moddate': '2026-01-05T22:01:47+09:00', 'title': '', 'trapped': '/False', 'source': './data/2026_gov.pdf', 'total_pages': 501, 'page': 4, 'page_label': '5'}
  내용 길이: 965 글자


### 온라인 PDF 로딩

URL에서 직접 PDF를 로드할 수도 있습니다.


In [13]:
# URL에서 PDF 로드 (예시)
# arxiv 논문을 직접 로드할 수 있습니다
url_example = "https://arxiv.org/pdf/2103.15348.pdf"

print("💡 온라인 PDF 로딩 방법:")
print(f"loader = PyPDFLoader('{url_example}')")
print("docs = loader.load()")
print("\n✅ 사용 사례:")
print("  - arXiv 논문 자동 수집")
print("  - 온라인 리포트 크롤링")
print("  - 웹에서 PDF 다운로드 없이 직접 처리")


💡 온라인 PDF 로딩 방법:
loader = PyPDFLoader('https://arxiv.org/pdf/2103.15348.pdf')
docs = loader.load()

✅ 사용 사례:
  - arXiv 논문 자동 수집
  - 온라인 리포트 크롤링
  - 웹에서 PDF 다운로드 없이 직접 처리


## 💡 핵심 정리

### PDF 로더 비교표

| 항목 | PyPDF | PyMuPDF | PDFPlumber |
|------|-------|---------|------------|
| **속도** | 중간 | ⚡ 가장 빠름 | 느림 |
| **정확도** | 기본 | 좋음 | 🎯 가장 정확 |
| **테이블 처리** | 약함 | 중간 | ✅ 우수 |
| **한글 지원** | 보통 | 좋음 | 좋음 |
| **OCR 지원** | ✅ 가능 | ❌ 불가 | ❌ 불가 |
| **설치 용이성** | ✅ 쉬움 | 보통 | 보통 |
| **권장 사용** | 일반 문서 | 대량 처리 | 복잡한 레이아웃 |

### 선택 결정 트리

```
PDF 문서가 있다
    ↓
[스캔 문서인가?]
    예 → PyPDF + OCR (extract_images=True)
    아니오 ↓
        [테이블/복잡한 레이아웃이 중요한가?]
            예 → PDFPlumber
            아니오 ↓
                [대량 문서 처리 (속도 중요)?]
                    예 → PyMuPDF
                    아니오 → PyPDF (기본 선택)
```

### 실전 팁

1. **처음에는 PyPDF로 시작**: 대부분의 경우에 충분합니다
2. **속도가 문제가 되면**: PyMuPDF로 전환
3. **텍스트 품질이 중요하면**: PDFPlumber 시도
4. **이미지 문서라면**: OCR 옵션 필수
5. **여러 로더 테스트**: 문서 특성에 따라 결과가 다를 수 있으므로 직접 비교


## 다음 단계

PDF 로더를 학습했으니, 다음은 다른 문서 형식을 다루는 로더들을 학습하겠습니다.

- **02-HWP-Loader**: 한글(HWP) 문서 로딩
- **03-CSV-Loader**: CSV 파일 로딩
- **04-Excel-Loader**: Excel 파일 로딩
- 그 외 Word, PowerPoint, JSON 등
